# Import

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi

In [2]:
spark = SparkSession.builder \
    .appName("Analisis-Tren-Waktu") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark

# Proses

In [3]:
df = spark.read.parquet(
    "hdfs://namenode:9000/data/processed/fintech/fraudTrain_clean")
df.printSchema()


root
 |-- cc_num: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- age: long (nullable = true)
 |-- distance: double (nullabl

# Analisis Tren Waktu

## Bulan

In [4]:
monthly = (
    df.groupBy("year", "month")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count")  
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraud_count") /
            col("total_transaction") * 100, 2
        )
    )
    .orderBy("year", "month")
)

monthly.show(25)

+----+-----+-----------------+------------------+-----------------+-----------+----------+
|year|month|total_transaction|      total_amount|       avg_amount|fraud_count|fraud_rate|
+----+-----+-----------------+------------------+-----------------+-----------+----------+
|2019|    1|            52525| 3759750.030000004| 71.5802004759639|        506|      0.96|
|2019|    2|            49866|3604662.2300000503|72.28697368948883|        517|      1.04|
|2019|    3|            70939| 5027886.400000043|70.87619504081032|        494|       0.7|
|2019|    4|            68078| 4761504.580000012|69.94189870442746|        376|      0.55|
|2019|    5|            72532| 5060903.319999984| 69.7747658964317|        408|      0.56|
|2019|    6|            86064| 6036397.420000003|70.13847160252838|        354|      0.41|
|2019|    7|            86596|  6044026.74000001|69.79568040094242|        331|      0.38|
|2019|    8|            87359| 6047288.649999909|69.22341888070959|        382|      0.44|

### Peak Low Fraud Month

In [5]:
peak_month = monthly.orderBy(
    col("fraud_count").desc())

low_month = monthly.orderBy(
    col("fraud_count").asc())

print("Peak Fraud Month All Periode")
peak_month.select(
    "year",
    "month",
    "fraud_count").show(1)

print("Low Fraud Month All Periode")
low_month.select(
    "year",
    "month",
    "fraud_count").show(1)


Peak Fraud Month All Periode
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2019|   12|        592|
+----+-----+-----------+
only showing top 1 row

Low Fraud Month All Periode
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2020|    4|        302|
+----+-----+-----------+
only showing top 1 row



#### Peak Low Fraud Month - 2019

In [6]:
monthly_2019 = monthly.filter(col("year") == 2019)

peak_month_2019 = (
    monthly_2019
    .orderBy(col("fraud_count").desc())
    .select("year", "month", "fraud_count")
    .limit(1)
)

low_month_2019 = (
    monthly_2019
    .orderBy(col("fraud_count").asc())
    .select("year", "month", "fraud_count")
    .limit(1)
)

print("Peak Month - 2019:")
peak_month_2019.show()

print("Low Month - 2019:")
low_month_2019.show()

Peak Month - 2019:
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2019|   12|        592|
+----+-----+-----------+

Low Month - 2019:
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2019|    7|        331|
+----+-----+-----------+



#### Peak Low Fraud Month - 2020

In [7]:
monthly_2020 = monthly.filter(col("year") == 2020)

peak_month_2020 = (
    monthly_2020
    .orderBy(col("fraud_count").desc())
    .select("year", "month", "fraud_count")
    .limit(1)
)

low_month_2020 = (
    monthly_2020
    .orderBy(col("fraud_count").asc())
    .select("year", "month", "fraud_count")
    .limit(1)
)

print("Peak Month - 2020:")
peak_month_2020.show()

print("Low Month - 2020:")
low_month_2020.show()

Peak Month - 2020:
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2020|    5|        527|
+----+-----+-----------+

Low Month - 2020:
+----+-----+-----------+
|year|month|fraud_count|
+----+-----+-----------+
|2020|    4|        302|
+----+-----+-----------+



### Peak Low Transaction Month

In [8]:
peak_month = monthly.orderBy(
    col("total_transaction").desc())

low_month = monthly.orderBy(
    col("total_transaction").asc())

print("Peak total_transaction Month")
peak_month.select(
    "year",
    "month",
    "total_transaction").show(1)

print("Low total_transaction Month")
low_month.select(
    "year",
    "month",
    "total_transaction").show(1)


Peak total_transaction Month
+----+-----+-----------------+
|year|month|total_transaction|
+----+-----+-----------------+
|2019|   12|           141060|
+----+-----+-----------------+
only showing top 1 row

Low total_transaction Month
+----+-----+-----------------+
|year|month|total_transaction|
+----+-----+-----------------+
|2020|    2|            47791|
+----+-----+-----------------+
only showing top 1 row



### Peak Low Amount Month

In [9]:
peak_month = monthly.orderBy(
    col("total_amount").desc())

low_month = monthly.orderBy(
    col("total_amount").asc())

print("Peak total_amount Month")
peak_month.select(
    "year",
    "month",
    "total_amount").show(1)

print("Low total_amount Month")
low_month.select(
    "year",
    "month",
    "total_amount").show(1)


Peak total_amount Month
+----+-----+-----------------+
|year|month|     total_amount|
+----+-----+-----------------+
|2019|   12|9918179.179999853|
+----+-----+-----------------+
only showing top 1 row

Low total_amount Month
+----+-----+-----------------+
|year|month|     total_amount|
+----+-----+-----------------+
|2020|    2|3369944.050000045|
+----+-----+-----------------+
only showing top 1 row



## Hari

In [10]:
day_of_week = (
    df.groupBy("day_of_week")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraud_count") /
            col("total_transaction") * 100, 2
        )
    )
    .orderBy("day_of_week")
)

day_of_week.show()

+-----------+-----------------+--------------------+-----------------+-----------+----------+
|day_of_week|total_transaction|        total_amount|       avg_amount|fraud_count|fraud_rate|
+-----------+-----------------+--------------------+-----------------+-----------+----------+
|          1|           250579|1.7379289420000087E7|69.35652796124211|       1216|      0.49|
|          2|           254282|1.7847845060000025E7|70.18917996555015|       1182|      0.46|
|          3|           160227|       1.124461784E7|70.17929462574972|        935|      0.58|
|          4|           131073|    9249541.86000002| 70.5678656931635|        859|      0.66|
|          5|           147285|1.0556820400000026E7|71.67614081542605|       1008|      0.68|
|          6|           152272| 1.076096963000005E7|70.66939181202092|       1079|      0.71|
|          7|           200957|1.4183344690000031E7|70.57900292102306|       1227|      0.61|
+-----------+-----------------+--------------------+--------

### Peak Low Fraud Day Of Week

In [11]:
peak_day_of_week = day_of_week.orderBy(
    col("fraud_count").desc())

low_day_of_week = day_of_week.orderBy(
    col("fraud_count").asc())

print("Peak Fraud day_of_week")
peak_day_of_week.select(
    "day_of_week",
    "fraud_count").show(2)

print("Low Fraud day_of_week")
low_day_of_week.select(
    "day_of_week",
    "fraud_count").show(2)


Peak Fraud day_of_week
+-----------+-----------+
|day_of_week|fraud_count|
+-----------+-----------+
|          7|       1227|
|          1|       1216|
+-----------+-----------+
only showing top 2 rows

Low Fraud day_of_week
+-----------+-----------+
|day_of_week|fraud_count|
+-----------+-----------+
|          4|        859|
|          3|        935|
+-----------+-----------+
only showing top 2 rows



### Peak Low Transaction Day Of Week

In [12]:
peak_day_of_week = day_of_week.orderBy(
    col("total_transaction").desc())

low_day_of_week = day_of_week.orderBy(
    col("total_transaction").asc())

print("Peak total_transaction day_of_week")
peak_day_of_week.select(
    "day_of_week",
    "total_transaction").show(2)

print("Low total_transaction day_of_week")
low_day_of_week.select(
    "day_of_week",
    "total_transaction").show(2)


Peak total_transaction day_of_week
+-----------+-----------------+
|day_of_week|total_transaction|
+-----------+-----------------+
|          2|           254282|
|          1|           250579|
+-----------+-----------------+
only showing top 2 rows

Low total_transaction day_of_week
+-----------+-----------------+
|day_of_week|total_transaction|
+-----------+-----------------+
|          4|           131073|
|          5|           147285|
+-----------+-----------------+
only showing top 2 rows



### Peak Low Amount Day Of Week

In [13]:
peak_day_of_week = day_of_week.orderBy(
    col("total_amount").desc())

low_day_of_week = day_of_week.orderBy(
    col("total_amount").asc())

print("Peak total_amount day_of_week")
peak_day_of_week.select(
    "day_of_week",
    "total_amount").show(2)

print("Low total_amount day_of_week")
low_day_of_week.select(
    "day_of_week",
    "total_amount").show(2)


Peak total_amount day_of_week
+-----------+--------------------+
|day_of_week|        total_amount|
+-----------+--------------------+
|          2|1.7847845060000025E7|
|          1|1.7379289420000087E7|
+-----------+--------------------+
only showing top 2 rows

Low total_amount day_of_week
+-----------+--------------------+
|day_of_week|        total_amount|
+-----------+--------------------+
|          4|    9249541.86000002|
|          5|1.0556820400000026E7|
+-----------+--------------------+
only showing top 2 rows



## Jam

In [14]:
hourly = (
    df.groupBy("hour")
    .agg(
        count("*").alias("total_transaction"),
        sum("amt").alias("total_amount"),
        avg("amt").alias("avg_amount"),
        sum("is_fraud").alias("fraud_count")
    )
    .withColumn(
        "fraud_rate",
        round(
            col("fraud_count") /
            col("total_transaction") * 100, 2
        )
    )
    .orderBy("hour")
)

hourly.show(24)

+----+-----------------+------------------+------------------+-----------+----------+
|hour|total_transaction|      total_amount|        avg_amount|fraud_count|fraud_rate|
+----+-----------------+------------------+------------------+-----------+----------+
|   0|            42502| 3364004.419999997| 79.14932050256452|        635|      1.49|
|   1|            42869| 3414097.520000005| 79.64024166647239|        658|      1.53|
|   2|            42656|3414411.1500000013|  80.0452726462866|        625|      1.47|
|   3|            42769|        3392361.19| 79.31822558395099|        609|      1.42|
|   4|            41863| 3185532.070000001| 76.09421374483436|         46|      0.11|
|   5|            42171|        3203208.63|  75.9576161343103|         60|      0.14|
|   6|            42300| 3201826.160000001|  75.6932898345154|         40|      0.09|
|   7|            42203| 3202941.929999997| 75.89370258038521|         56|      0.13|
|   8|            42505|3235989.0799999954|  76.131962

### Peak Low Fraud Hour

In [15]:
peak_hour = hourly.orderBy(
    col("fraud_count").desc())

low_hour = hourly.orderBy(
    col("fraud_count").asc())

print("Peak Fraud hour")
peak_hour.select(
    "hour",
    "fraud_count").show(2)

print("Low Fraud hour")
low_hour.select(
    "hour",
    "fraud_count").show(2)


Peak Fraud hour
+----+-----------+
|hour|fraud_count|
+----+-----------+
|  22|       1931|
|  23|       1904|
+----+-----------+
only showing top 2 rows

Low Fraud hour
+----+-----------+
|hour|fraud_count|
+----+-----------+
|   6|         40|
|  10|         40|
+----+-----------+
only showing top 2 rows



### Peak Low Transaction Hour

In [16]:
peak_hour = hourly.orderBy(
    col("total_transaction").desc())

low_hour = hourly.orderBy(
    col("total_transaction").asc())

print("Peak total_transaction hour")
peak_hour.select(
    "hour",
    "total_transaction").show(2)

print("Low total_transaction hour")
low_hour.select(
    "hour",
    "total_transaction").show(2)


Peak total_transaction hour
+----+-----------------+
|hour|total_transaction|
+----+-----------------+
|  23|            67104|
|  22|            66982|
+----+-----------------+
only showing top 2 rows

Low total_transaction hour
+----+-----------------+
|hour|total_transaction|
+----+-----------------+
|   4|            41863|
|  11|            42082|
+----+-----------------+
only showing top 2 rows



### Peak Low Amount Hour

In [17]:
peak_hour = hourly.orderBy(
    col("total_amount").desc())

low_hour = hourly.orderBy(
    col("total_amount").asc())

print("Peak total_amount hour")
peak_hour.select(
    "hour",
    "total_amount").show(2)

print("Low total_amount hour")
low_hour.select(
    "hour",
    "total_amount").show(2)


Peak total_amount hour
+----+-----------------+
|hour|     total_amount|
+----+-----------------+
|  22|5408217.790000018|
|  23| 5308144.31999999|
+----+-----------------+
only showing top 2 rows

Low total_amount hour
+----+-----------------+
|hour|     total_amount|
+----+-----------------+
|   9|3170824.339999998|
|   4|3185532.070000001|
+----+-----------------+
only showing top 2 rows



# Output

In [18]:
monthly.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/monthly")
day_of_week.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/day_of_week")
hourly.write.mode("overwrite").parquet("hdfs://namenode:9000/data/analytics/fintech/hourly")